# 05 · Integrating Orientation

### Recap & why now
A gyroscope reports **body-frame angular velocity**: three numbers, a hundred times a
second. Turning that stream into an orientation is the job of one small differential
equation, and it is the last piece of machinery before the physics starts.

It is also where a subtle failure lives. The equation is exact in continuous time and
slightly wrong in discrete time, and the error accumulates until the quaternion is no
longer a rotation at all.

### Learning objectives
1. Write $\dot q = \tfrac{1}{2} q \otimes \omega_q$ and say what each part is.
2. Integrate it numerically and compare against a known closed-form answer.
3. Watch $|q|$ **drift** away from 1, and quantify what that costs.
4. Fix it with one line, and measure the accuracy that remains.
5. Explain why this equation is linear, and why that matters.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

def quat_from_rotmat(R):
    """Rotation matrix -> quaternion. Four branches, so we never divide by a small number."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)

def axis_angle_to_quat(axis, angle):
    """Build a quaternion from 'rotate by `angle` about `axis`' — the geometric reading."""
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(axis*np.sin(angle/2))])

def quat_rotate(q, v):
    """Rotate v from the body frame into the world frame, using the sandwich product."""
    return quat_multiply(quat_multiply(q, np.array([0.0, *v])), quat_conjugate(q))[1:]

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

## 1 · The kinematic equation

$$\dot q = \tfrac{1}{2}\, q \otimes \omega_q,
\qquad \omega_q = [\,0,\; \omega_x,\; \omega_y,\; \omega_z\,]$$

The angular velocity is padded with a zero to make it a quaternion, then multiplied on
the **right** — which is what puts it in the body frame, exactly where a gyro measures
it.

Two features matter. It is **linear in $q$**: no trigonometry, no division, nothing
waiting to explode — compare that with Notebook 03's $1/\cos\theta$. And it does
**not** preserve $|q| = 1$ once you discretise it.

In [ ]:
def quat_derivative(q, omega_body):
    """dq/dt for a body-frame angular velocity. The half is not optional."""
    return 0.5*quat_multiply(q, np.array([0.0, *omega_body]))   # omega padded with a leading zero.

q0 = euler_to_quat(0.1, -0.05, 0.2)
omega = np.array([1.5, -0.8, 0.6])                 # rad/s, in the body frame.

print("q      =", np.round(q0, 4))
print("dq/dt  =", np.round(quat_derivative(q0, omega), 4))
print("\nnotice: the derivative is a linear function of q, so a doubling of q doubles dq/dt.")
print("Every term is a multiplication and an addition — nothing here can divide by zero.")

## 2 · Integrating it, and checking

For a **constant** angular velocity there is a closed-form answer: the drone turns
through $|\omega| t$ about the fixed axis $\omega/|\omega|$. That gives us something to
check the integration against, which is worth having before we trust it for ten
notebooks.

In [ ]:
def integrate(q_start, omega_body, T_end, dt, normalise=True):
    """March the quaternion forward, optionally renormalising each step."""
    q = np.asarray(q_start, float).copy(); norms = [np.linalg.norm(q)]
    for _ in range(int(T_end/dt)):
        q = q + dt*quat_derivative(q, omega_body)  # Plain Euler: one derivative, one step.
        if normalise:
            q = quat_normalize(q)                  # The one line this notebook is about.
        norms.append(np.linalg.norm(q))
    return q, np.array(norms)

def angle_between(q1, q2):
    """Smallest rotation angle between two attitudes, in degrees."""
    d = abs(float(quat_normalize(q1) @ quat_normalize(q2)))
    return np.degrees(2*np.arccos(min(1.0, d)))

T_end = 20.0
exact = quat_multiply(q0, axis_angle_to_quat(omega, np.linalg.norm(omega)*T_end))   # The closed form.

print("  step size      attitude error after %.0f s" % T_end)
for dt in (0.02, 0.01, 0.005, 0.002):
    q_end, _ = integrate(q0, omega, T_end, dt)
    print("   %.3f s %20.3f°" % (dt, angle_between(q_end, exact)))
print("\nHalving the step halves the error — first-order accuracy, exactly as expected from")
print("plain Euler. Notebook 06 upgrades to RK4 and the numbers improve dramatically.")

## 3 · The drift, and what it costs

Now remove the renormalisation and watch $|q|$ wander. The discrete step moves along
the *tangent* to the unit sphere and lands slightly outside it, every time.

The error per step is tiny. The accumulation is not — and the consequence is specific:
`quat_to_rotmat` on an over-long quaternion produces a matrix that **scales** as well as
rotates, so the drone silently gains thrust it does not have.

In [ ]:
_, norms_drift = integrate(q0, omega, T_end, 0.01, normalise=False)   # No safety net.
_, norms_clean = integrate(q0, omega, T_end, 0.01, normalise=True)

fig, ax = plt.subplots(figsize=(7.4, 3.0))
t = np.arange(len(norms_drift))*0.01
ax.plot(t, norms_drift, color="C3", lw=2, label="no normalisation")
ax.plot(t, norms_clean, color="C0", lw=2, label="normalised every step")
ax.axhline(1.0, color="k", ls=":", lw=1)
ax.set_xlabel("time [s]"); ax.set_ylabel("|q|"); ax.legend(fontsize=9)
ax.set_title("Drifting off the unit sphere")
plt.show()

q_drift, _ = integrate(q0, omega, T_end, 0.01, normalise=False)
scale = np.linalg.norm(quat_to_rotmat(q_drift/1.0) @ np.array([0, 0, 1.0]))
print("after %.0f s: |q| = %.4f without normalisation, %.6f with." % (T_end, norms_drift[-1], norms_clean[-1]))
print("an over-long quaternion scales vectors by |q|^2 = %.4f — so a thrust command of 9.81 N" %
      norms_drift[-1]**2)
print("would be applied as %.2f N. The drone climbs for no reason anyone can find in the code." %
      (9.81*norms_drift[-1]**2))

## 4 · One line, and the accuracy that remains

`q = quat_normalize(q)` after every step removes the entire failure mode. What it does
**not** remove is the ordinary integration error — the drift in $|q|$ and the error in
the attitude are two different things, and only one of them is cured by normalising.

The table below separates them, which is worth doing once so the two are never confused.

In [ ]:
print("  step size    |q| drift (no norm)    attitude error (with norm)")
for dt in (0.02, 0.01, 0.005, 0.002):
    _, nd = integrate(q0, omega, T_end, dt, normalise=False)
    q_end, _ = integrate(q0, omega, T_end, dt, normalise=True)
    print("   %.3f s %18.6f %24.3f°" % (dt, nd[-1] - 1.0, angle_between(q_end, exact)))

print("\nBoth improve with a smaller step, and they are independent problems: normalisation")
print("fixes the first exactly and the second not at all. Use both — normalise every step AND")
print("integrate accurately — because they are cheap and they fail in different ways.")

## 🧪 Try it yourself

**E1.** The equation multiplies $\omega$ on the **right** of $q$. What would change if it
were on the left, and which convention would that correspond to?

**E2.** Integrate $\omega = [0, 0, 2]$ for 3 seconds and confirm the yaw is what you
expect. Then try $\omega = [2, 0, 2]$ and check whether naive intuition still holds.

In [ ]:
# --- Solution E1 ---
q_right, _ = integrate(q0, omega, 1.0, 0.001)
q_left = q0.copy()
for _ in range(1000):
    q_left = quat_normalize(q_left + 0.001*0.5*quat_multiply(np.array([0.0, *omega]), q_left))
print("E1: right-multiplied and left-multiplied end %.1f° apart after one second." %
      angle_between(q_right, q_left))
print("    Right multiplication treats omega as measured in the BODY frame, which is what a")
print("    strapped-down gyroscope reports. Left multiplication would treat it as a WORLD-frame")
print("    rate — meaningful, but not what any sensor on the drone measures. Choosing the wrong")
print("    side gives a simulator that behaves correctly only while the drone is near level.")

# --- Solution E2 ---
q_yaw, _ = integrate(np.array([1.0, 0, 0, 0]), [0, 0, 2.0], 3.0, 0.001)
print("\nE2: omega = [0,0,2] for 3 s -> roll/pitch/yaw =", np.round(np.degrees(quat_to_euler(q_yaw)), 2))
print("    expected yaw = 2 rad/s x 3 s = 6 rad = %.1f°, which wraps to %.1f° ✔" %
      (np.degrees(6.0), np.degrees(6.0) - 360))

q_mix, _ = integrate(np.array([1.0, 0, 0, 0]), [2.0, 0, 2.0], 3.0, 0.001)
print("\n    omega = [2,0,2] for 3 s -> roll/pitch/yaw =", np.round(np.degrees(quat_to_euler(q_mix)), 2))
print("    Naive intuition says 'roll 343.8°, yaw 343.8°, pitch 0' — and it is wrong, because the")
print("    body axes are themselves rotating. After some roll, the body z is no longer world z, so")
print("    the yaw component starts tilting the drone. Angular velocity does not integrate into")
print("    Euler angles. It integrates into a quaternion, which is the entire reason we use one.")

## 🚁 Mini-project: a tumbling drone

Integrate a constant angular velocity about a slanted axis and animate the result. The
drone tumbles steadily and the quaternion stays exactly unit-length throughout — the
combination the next five notebooks are built on.

In [ ]:
omega_tumble = np.array([1.2, 0.6, 0.9])           # A rotation about no particular axis.
dt, n = 0.02, 200
q = np.array([1.0, 0.0, 0.0, 0.0]); poses, norms = [], []
for k in range(n):
    q = quat_normalize(q + dt*quat_derivative(q, omega_tumble))
    poses.append(q.copy()); norms.append(np.linalg.norm(q))

fig = plt.figure(figsize=(9.8, 4.4))

def frame(k):
    fig.clf()
    ax = fig.add_subplot(121, projection="3d")
    draw_quad(ax, [0, 0, 0], poses[k], scale=2.2)
    ax.quiver(0, 0, 0, *(omega_tumble/np.linalg.norm(omega_tumble)*1.1),
              color="C4", lw=2, arrow_length_ratio=0.15)      # The fixed rotation axis.
    set_3d(ax, (-1.2, 1.2), (-1.2, 1.2), (-1.0, 1.2))
    ax.set_title("t = %4.2f s" % (k*dt), fontsize=10); ax.view_init(elev=18, azim=-62)
    ax2 = fig.add_subplot(122)
    rpy = np.degrees(np.array([quat_to_euler(p) for p in poses[:k+1]]))
    for i, lbl in enumerate(["roll", "pitch", "yaw"]):
        ax2.plot(np.arange(k+1)*dt, rpy[:, i], lw=1.5, label=lbl)
    ax2.set_xlim(0, n*dt); ax2.set_ylim(-200, 200); ax2.legend(fontsize=8)
    ax2.set_xlabel("time [s]"); ax2.set_ylabel("angle [deg]")
    ax2.set_title("|q| = %.10f" % norms[k], fontsize=9)
    return []

anim = animation.FuncAnimation(fig, frame, frames=n, interval=45, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** This one equation is the core of every strapdown inertial
> navigation system: gyroscope in, orientation out, a thousand times a second. Real
> implementations add a few refinements — higher-order integration, and coning
> compensation for the fact that the angular velocity itself rotates within a step — but
> the renormalisation is universal, and it is usually the first thing to check when an
> attitude estimate starts behaving strangely.

**Where next.** We can describe and evolve an orientation. Notebook 06 gives it physics:
thrust, gravity, torque and inertia, assembled into the thirteen numbers the simulator
carries.